# Deep lob

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import torch.optim as optim

In [2]:


class EncoderDecoderModel(nn.Module):
    def __init__(self, latent_dim):
        super(EncoderDecoderModel, self).__init__()

        # Encoder layers (convolutional)
        self.conv_first1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=(1, 2), stride=(1, 2)),
            nn.LeakyReLU(0.01),
            nn.Conv2d(32, 32, kernel_size=(4, 1), padding='same'),
            nn.LeakyReLU(0.01),
            nn.Conv2d(32, 32, kernel_size=(4, 1), padding='same'),
            nn.LeakyReLU(0.01),
            nn.Conv2d(32, 32, kernel_size=(1, 2), stride=(1, 2)),
            nn.LeakyReLU(0.01),
            nn.Conv2d(32, 32, kernel_size=(4, 1), padding='same'),
            nn.LeakyReLU(0.01),
            nn.Conv2d(32, 32, kernel_size=(4, 1), padding='same'),
            nn.LeakyReLU(0.01),
            nn.Conv2d(32, 32, kernel_size=(1, 10)),
            nn.LeakyReLU(0.01),
            nn.Conv2d(32, 32, kernel_size=(4, 1), padding='same'),
            nn.LeakyReLU(0.01),
            nn.Conv2d(32, 32, kernel_size=(4, 1), padding='same'),
            nn.LeakyReLU(0.01)
        )

        # Inception module
        self.convsecond_1 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=(1, 1), padding='same'),
            nn.LeakyReLU(0.01),
            nn.Conv2d(64, 64, kernel_size=(3, 1), padding='same'),
            nn.LeakyReLU(0.01)
        )

        self.convsecond_2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=(1, 1), padding='same'),
            nn.LeakyReLU(0.01),
            nn.Conv2d(64, 64, kernel_size=(5, 1), padding='same'),
            nn.LeakyReLU(0.01)
        )

        self.convsecond_3 = nn.Sequential(
            nn.MaxPool2d(kernel_size=(3, 1), stride=(1, 1), padding='same'),
            nn.Conv2d(32, 64, kernel_size=(1, 1), padding='same'),
            nn.LeakyReLU(0.01)
        )

        # Decoder layers (LSTM-based)
        self.encoder_lstm = nn.LSTM(input_size=64, hidden_size=latent_dim, batch_first=True)
        self.decoder_lstm = nn.LSTM(input_size=3, hidden_size=latent_dim, batch_first=True)
        self.decoder_dense = nn.Linear(latent_dim + 64, 3)

    def forward(self, input_train, decoder_inputs):
        # Encoder
        conv_first1_out = self.conv_first1(input_train)
        convsecond_1_out = self.convsecond_1(conv_first1_out)
        convsecond_2_out = self.convsecond_2(conv_first1_out)
        convsecond_3_out = self.convsecond_3(conv_first1_out)
        convsecond_output = torch.cat((convsecond_1_out, convsecond_2_out, convsecond_3_out), dim=1)
        conv_reshape = convsecond_output.view(convsecond_output.size(0), convsecond_output.size(2), -1)

        # LSTM encoder
        encoder_outputs, (state_h, state_c) = self.encoder_lstm(conv_reshape)

        # Initial decoder inputs
        decoder_inputs = decoder_inputs.unsqueeze(1)  # (batch_size, 1, 3)

        all_outputs = []
        inputs = torch.cat((decoder_inputs, encoder_outputs.unsqueeze(1)), dim=2)

        # Decoder LSTM loop
        for _ in range(5):
            outputs, (state_h, state_c) = self.decoder_lstm(inputs, (state_h, state_c))
            outputs = self.decoder_dense(torch.cat((outputs, encoder_outputs.unsqueeze(1)), dim=2))
            all_outputs.append(outputs)
            inputs = torch.cat((outputs, encoder_outputs.unsqueeze(1)), dim=2)

        # Concatenate all predictions
        decoder_outputs = torch.cat(all_outputs, dim=1)

        return decoder_outputs

# Creating an instance of the model
latent_dim = 64  # Set your desired latent dimension
model = EncoderDecoderModel(latent_dim)


In [3]:
# Define parameters
T = 50
epochs = 50
batch_size = 32
n_hidden = 64

# Create synthetic data
# Assuming input_train and decoder_inputs are randomly generated tensors
input_train = torch.randn(batch_size, 1, 50, 40, 1)  # batch_size x channels x height x width x extra_dim
decoder_inputs = torch.randn(batch_size, 1, 3)  # batch_size x seq_length x features

# Reshape input_train to remove the extra dimension
input_train = input_train.squeeze(-1)  # Remove the last dimension (extra_dim)

In [5]:

# Creating an instance of the model
model = EncoderDecoderModel(n_hidden)

# Loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(input_train, decoder_inputs)
    loss = criterion(outputs, torch.randn(batch_size, 5, 3))  # Example random target, adjust as per your actual target
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

print('Training finished!')


RuntimeError: Expected 3D (unbatched) or 4D (batched) input to conv2d, but got input of size: [32, 1, 50, 40, 1]